# Setup Lamna Healthcare Ontology

This notebook automates the setup for Fabric IQ ontology labs. It creates all required infrastructure, loads sample data, and builds the complete ontology.

**The notebook is idempotent** — running it multiple times produces the same result. If infrastructure exists, it reuses it; if tables exist, it overwrites them; if the ontology exists, it deletes and recreates it. You can run it as many times as needed without causing duplicates or errors.

- Creates or reuses lakehouse (`LamnaHealthcareLH`) and eventhouse (`LamnaHealthcareEH`)
- Writes 5 lakehouse tables with hospital operations data
- Ingests time-series vital signs readings to eventhouse
- Builds `LamnaHealthcareOntology` with 5 entity types + 4 relationships

## Step 0: Get or Create Infrastructure

Checks for existing lakehouse and eventhouse. Creates them if they don't exist.

> **Note**: The first cell installs a required package and may show dependency warnings. These warnings are normal in Fabric notebook environments and don't affect functionality—you can safely ignore them.

In [ ]:
%pip install semantic-link --quiet --disable-pip-version-check

import requests, json, base64, time, random, uuid
import sempy.fabric as fabric
from notebookutils import mssparkutils
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import to_timestamp

workspace_id = fabric.get_workspace_id()
token = mssparkutils.credentials.getToken("pbi")
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
print(f"Workspace ID: {workspace_id}\n")

# ── Lakehouse ──────────────────────────────────────────────────────────────────
print("1️⃣  Checking for LamnaHealthcareLH...")
resp = requests.get(
    f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/lakehouses",
    headers=headers
)
lakehouses = resp.json().get("value", [])
lh = next((l for l in lakehouses if l["displayName"] == "LamnaHealthcareLH"), None)

if lh:
    lakehouse_id = lh["id"]
    print(f"   ✓ Found existing LamnaHealthcareLH (id: {lakehouse_id})")
else:
    print("   Creating LamnaHealthcareLH...")
    lakehouse_id = fabric.create_lakehouse(
        display_name="LamnaHealthcareLH",
        description="Healthcare lakehouse for Fabric IQ lab",
        enable_schema=False
    )
    print(f"   ✓ Created LamnaHealthcareLH (id: {lakehouse_id})")

lakehouse_tables_path = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Tables"

# ── Eventhouse ─────────────────────────────────────────────────────────────────
print("\n2️⃣  Checking for LamnaHealthcareEH...")
resp = requests.get(
    f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/eventhouses",
    headers=headers
)
eventhouses = resp.json().get("value", [])
eh = next((e for e in eventhouses if e["displayName"] == "LamnaHealthcareEH"), None)

if eh:
    eventhouse_id = eh["id"]
    print(f"   ✓ Found existing LamnaHealthcareEH (id: {eventhouse_id})")
else:
    print("   Creating LamnaHealthcareEH...")
    resp = requests.post(
        f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/eventhouses",
        headers=headers,
        json={"displayName": "LamnaHealthcareEH"}
    )
    eventhouse_id = resp.json()["id"]
    print(f"   ✓ Created LamnaHealthcareEH (id: {eventhouse_id})")
    print("   ⏳ Waiting for eventhouse to initialize...")
    time.sleep(10)

# Get eventhouse query URI
resp = requests.get(
    f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/eventhouses/{eventhouse_id}",
    headers=headers
)
query_uri = resp.json()["properties"]["queryServiceUri"]
print(f"\n   Eventhouse query URI: {query_uri}")

spark = SparkSession.builder.getOrCreate()

print(f"\n{'='*60}")
print(f"✅ Infrastructure ready!")
print(f"   Lakehouse : LamnaHealthcareLH  ({lakehouse_id})")
print(f"   Eventhouse: LamnaHealthcareEH  ({eventhouse_id})")
print(f"   OneLake   : {lakehouse_tables_path}")
print(f"{'='*60}")

## Step 1: Write Lakehouse Tables

Writes all 5 healthcare tables to the lakehouse (overwrite mode — safe to re-run).

In [ ]:
print("Writing lakehouse tables...\n")

# ── Hospitals ──────────────────────────────────────────────────────────────────
hospitals_data = [(1, "Lamna Healthcare Regional Medical Center", "Seattle", "WA")]
hospitals_schema = StructType([
    StructField("HospitalId",   IntegerType(), False),
    StructField("HospitalName", StringType(),  False),
    StructField("City",         StringType(),  False),
    StructField("State",        StringType(),  False)
])
spark.createDataFrame(hospitals_data, hospitals_schema) \
     .write.mode("overwrite").format("delta").save(f"{lakehouse_tables_path}/Hospitals")
print("✓ Hospitals          (1 row)")

# ── Departments ────────────────────────────────────────────────────────────────
departments_data = [
    (1, "Intensive Care Unit",    1, 3),
    (2, "Emergency Department",   1, 1),
    (3, "Surgical Services",      1, 2)
]
departments_schema = StructType([
    StructField("DepartmentId",   IntegerType(), False),
    StructField("DepartmentName", StringType(),  False),
    StructField("HospitalId",     IntegerType(), False),
    StructField("Floor",          IntegerType(), False)
])
spark.createDataFrame(departments_data, departments_schema) \
     .write.mode("overwrite").format("delta").save(f"{lakehouse_tables_path}/Departments")
print("✓ Departments        (3 rows)")

# ── Rooms ──────────────────────────────────────────────────────────────────────
rooms_data = [
    (1,  "ICU-301", 1, "Critical Care"),
    (2,  "ICU-302", 1, "Critical Care"),
    (3,  "ICU-303", 1, "Critical Care"),
    (4,  "ER-101",  2, "Emergency"),
    (5,  "ER-102",  2, "Emergency"),
    (6,  "ER-103",  2, "Emergency"),
    (7,  "SUR-201", 3, "Post-Op"),
    (8,  "SUR-202", 3, "Post-Op"),
    (9,  "SUR-203", 3, "Post-Op"),
    (10, "SUR-204", 3, "Post-Op")
]
rooms_schema = StructType([
    StructField("RoomId",       IntegerType(), False),
    StructField("RoomNumber",   StringType(),  False),
    StructField("DepartmentId", IntegerType(), False),
    StructField("RoomType",     StringType(),  False)
])
spark.createDataFrame(rooms_data, rooms_schema) \
     .write.mode("overwrite").format("delta").save(f"{lakehouse_tables_path}/Rooms")
print("✓ Rooms              (10 rows)")

# ── Patients ───────────────────────────────────────────────────────────────────
patients_data = [
    (1001, "Kerry",  "Allen",    "1965-03-15", "2026-03-25", 1),
    (1002, "Casey",  "Morgan",   "1978-07-22", "2026-03-26", 2),
    (1003, "Elijah", "Thompson", "1952-11-08", "2026-03-27", 3),
    (1004, "Priya",  "Desai",    "1988-05-19", "2026-03-28", 7),
    (1005, "Maya",   "Robinson", "1971-09-30", "2026-03-29", 8)
]
patients_schema = StructType([
    StructField("PatientId",     IntegerType(),   False),
    StructField("FirstName",     StringType(),    False),
    StructField("LastName",      StringType(),    False),
    StructField("DateOfBirth",   TimestampType(), False),
    StructField("AdmissionDate", TimestampType(), False),
    StructField("CurrentRoomId", IntegerType(),   False)
])
df_patients = spark.createDataFrame(patients_data, StructType([
    StructField("PatientId",     IntegerType(), False),
    StructField("FirstName",     StringType(),  False),
    StructField("LastName",      StringType(),  False),
    StructField("DateOfBirth",   StringType(),  False),
    StructField("AdmissionDate", StringType(),  False),
    StructField("CurrentRoomId", IntegerType(), False)
]))
df_patients = df_patients \
    .withColumn("DateOfBirth", to_timestamp("DateOfBirth", "yyyy-MM-dd")) \
    .withColumn("AdmissionDate", to_timestamp("AdmissionDate", "yyyy-MM-dd"))
df_patients.write.mode("overwrite").format("delta").save(f"{lakehouse_tables_path}/Patients")
print("✓ Patients           (5 rows)")

# ── VitalSignEquipment ─────────────────────────────────────────────────────────
equipment_data = [
    ("VS-1001", 1001, "Continuous Monitoring", "2026-03-25"),
    ("VS-1002", 1002, "Continuous Monitoring", "2026-03-26"),
    ("VS-1003", 1003, "Continuous Monitoring", "2026-03-27"),
    ("VS-1004", 1004, "Continuous Monitoring", "2026-03-28"),
    ("VS-1005", 1005, "Continuous Monitoring", "2026-03-29")
]
df_equipment = spark.createDataFrame(equipment_data, StructType([
    StructField("EquipmentId",          StringType(),  False),
    StructField("PatientId",            IntegerType(), False),
    StructField("EquipmentType",        StringType(),  False),
    StructField("MonitoringStartDate",  StringType(),  False)
]))
df_equipment = df_equipment.withColumn("MonitoringStartDate", to_timestamp("MonitoringStartDate", "yyyy-MM-dd"))
df_equipment.write.mode("overwrite").format("delta").save(f"{lakehouse_tables_path}/VitalSignEquipment")
print("✓ VitalSignEquipment (5 rows)")

print("\n✅ All lakehouse tables written!")

## Step 2: Ingest VitalSignsReadings to Eventhouse

Creates the eventhouse table and ingests time-series vital signs data (20 rows). Skips if data already exists.

In [ ]:
db = "LamnaHealthcareEH"

kql_token = mssparkutils.credentials.getToken(query_uri)
kql_headers = {"Authorization": f"Bearer {kql_token}", "Content-Type": "application/json"}

def kql_mgmt(command):
    resp = requests.post(f"{query_uri}/v1/rest/mgmt", headers=kql_headers, json={"db": db, "csl": command})
    resp.raise_for_status()
    return resp.json()

def kql_query(query):
    resp = requests.post(f"{query_uri}/v1/rest/query", headers=kql_headers, json={"db": db, "csl": query})
    resp.raise_for_status()
    tables = resp.json().get("Tables", [])
    return tables[0]["Rows"] if tables else []

# Create table
kql_mgmt("""
.create-merge table VitalSignsReadings (
    ReadingId:        int,
    EquipmentId:      string,
    Timestamp:        datetime,
    HeartRate:        int,
    OxygenSaturation: int,
    RespiratoryRate:  int
)
""")
print("✓ VitalSignsReadings table created / verified")

# Check if data exists
rows = kql_query("VitalSignsReadings | count")
existing_count = rows[0][0] if rows else 0

if existing_count > 0:
    print(f"ℹ️  VitalSignsReadings already has {existing_count} rows — skipping ingestion")
else:
    readings_csv = """1,VS-1001,2026-04-01T03:00:00Z,70,95,13
2,VS-1001,2026-04-01T03:05:00Z,72,95,13
3,VS-1001,2026-04-01T03:10:00Z,71,96,14
4,VS-1002,2026-04-01T03:00:00Z,68,97,12
5,VS-1002,2026-04-01T03:05:00Z,67,97,12
6,VS-1002,2026-04-01T03:10:00Z,69,98,13
7,VS-1003,2026-04-01T03:00:00Z,65,95,13
8,VS-1003,2026-04-01T03:05:00Z,66,94,14
9,VS-1003,2026-04-01T03:10:00Z,67,96,14
10,VS-1004,2026-04-01T03:00:00Z,95,98,16
11,VS-1004,2026-04-01T03:05:00Z,98,98,16
12,VS-1004,2026-04-01T03:10:00Z,100,97,17
13,VS-1005,2026-04-01T03:00:00Z,78,96,15
14,VS-1005,2026-04-01T03:05:00Z,77,97,15
15,VS-1005,2026-04-01T03:10:00Z,79,97,14"""
    kql_mgmt(f".ingest inline into table VitalSignsReadings <|\n{readings_csv}")
    print("✓ Ingested 15 rows into VitalSignsReadings")

print("\n✅ Eventhouse step complete!")

## Step 3: Build Ontology Definition

Defines 5 entity types with data bindings and 4 relationship types:

**Entities:** Hospitals, Departments, Rooms, Patients, VitalSignEquipment

**VitalSignEquipment has two bindings:**
- Static: lakehouse table (EquipmentId, PatientId, EquipmentType, MonitoringStartDate)
- Time-series: eventhouse VitalSignsReadings table (ReadingId, Timestamp, HeartRate, OxygenSaturation, RespiratoryRate)

**Relationships:** Departments→Hospitals, Rooms→Departments, Patients→Rooms, VitalSignEquipment→Patients

In [ ]:
def det_id(name):
    """Deterministic ID from string — stable across re-runs."""
    random.seed(name)
    return random.randint(1_000_000_000_000, 9_999_999_999_999)

def b64(obj):
    return base64.b64encode(json.dumps(obj).encode()).decode()

# Entity IDs
hosp_id  = det_id("Hospitals")
dept_id  = det_id("Departments")
room_id  = det_id("Rooms")
pat_id   = det_id("Patients")
equip_id = det_id("VitalSignEquipment")

# Property IDs
hosp_props  = {k: det_id(f"Hospitals_{k}") for k in ["HospitalId", "HospitalName", "City", "State"]}
dept_props  = {k: det_id(f"Departments_{k}") for k in ["DepartmentId", "DepartmentName", "HospitalId", "Floor"]}
room_props  = {k: det_id(f"Rooms_{k}") for k in ["RoomId", "RoomNumber", "DepartmentId", "RoomType"]}
pat_props   = {k: det_id(f"Patients_{k}") for k in ["PatientId", "FirstName", "LastName", "DateOfBirth", "AdmissionDate", "CurrentRoomId"]}
equip_props = {k: det_id(f"VitalSignEquipment_{k}") for k in ["EquipmentId", "PatientId", "EquipmentType", "MonitoringStartDate"]}
# Time-series properties for VitalSignEquipment (from eventhouse VitalSignsReadings)
# Only include actual measurements, not row identifiers (ReadingId is excluded)
ts_props = {k: det_id(f"VitalSignEquipment_TS_{k}") for k in ["Timestamp", "HeartRate", "OxygenSaturation", "RespiratoryRate"]}

# Relationship IDs
rel_dept_hosp = det_id("Rel_Departments_inHospital")
rel_room_dept = det_id("Rel_Rooms_inDepartment")
rel_pat_room  = det_id("Rel_Patients_admittedTo")
rel_equip_pat = det_id("Rel_VitalSignEquipment_assignedToPatient")

def prop_entry(k, pid, vt):
    return {"id": str(pid), "name": k, "valueType": vt, "redefines": None, "baseTypeNamespaceType": None}

def entity_def(eid, name, id_prop, display_prop, props_dict, value_types, ts_value_types=None):
    return {
        "id": str(eid), "namespace": "usertypes", "namespaceType": "Custom",
        "name": name, "visibility": "Visible",
        "entityIdParts": [str(props_dict[id_prop])],
        "displayNamePropertyId": str(props_dict[id_prop]),
        "baseEntityTypeId": None,
        "properties": [prop_entry(k, props_dict[k], vt) for k, vt in value_types.items()],
        "timeseriesProperties": [prop_entry(k, pid, vt) for k, (pid, vt) in (ts_value_types or {}).items()]
    }

def lh_binding(entity_props, table_name):
    bid = str(uuid.uuid4())
    return {
        "id": bid,
        "dataBindingConfiguration": {
            "dataBindingType": "NonTimeSeries",
            "propertyBindings": [{"sourceColumnName": col, "targetPropertyId": str(pid)} for col, pid in entity_props.items()],
            "sourceTableProperties": {
                "sourceType": "LakehouseTable",
                "workspaceId": workspace_id,
                "itemId": lakehouse_id,
                "sourceTableName": table_name
            }
        }
    }

def kql_ts_binding(ts_props_map, kql_table, timestamp_col, match_col, match_prop_id):
    """Time-series binding for VitalSignEquipment from eventhouse VitalSignsReadings."""
    bid = str(uuid.uuid4())
    return {
        "id": bid,
        "dataBindingConfiguration": {
            "dataBindingType": "TimeSeries",
            "timestampColumnName": timestamp_col,
            "propertyBindings": [
                # Time-series measurements (ALL timeseries properties INCLUDING timestamp)
                *[{"sourceColumnName": col, "targetPropertyId": str(pid)} for col, pid in ts_props_map.items()],
                # Match key MUST be LAST: EquipmentId column → static EquipmentId property (for contextualization)
                {"sourceColumnName": match_col, "targetPropertyId": str(match_prop_id)}
            ],
            "sourceTableProperties": {
                "sourceType": "KustoTable",
                "workspaceId": workspace_id,
                "itemId": eventhouse_id,
                "clusterUri": query_uri,
                "databaseName": "LamnaHealthcareEH",
                "sourceTableName": kql_table
            }
        }
    }

def rel_def(rid, name, src_id, tgt_id):
    return {
        "id": str(rid), "namespace": "usertypes", "namespaceType": "Custom",
        "name": name,
        "source": {"entityTypeId": str(src_id)},
        "target": {"entityTypeId": str(tgt_id)}
    }

def rel_contextualization(table_name, src_col, src_prop_id, tgt_col, tgt_prop_id):
    """Relationship contextualization - binds relationship to data using Contextualizations API.
    
    Args:
        table_name: Table containing both source and target identifying columns
        src_col: Column name in table that identifies the source entity
        src_prop_id: Property ID of the source entity's key
        tgt_col: Column name in table that identifies the target entity
        tgt_prop_id: Property ID of the target entity's key
    """
    cid = str(uuid.uuid4())
    return {
        "id": cid,
        "dataBindingTable": {
            "workspaceId": workspace_id,
            "itemId": lakehouse_id,
            "sourceTableName": table_name,
            "sourceType": "LakehouseTable"
        },
        "sourceKeyRefBindings": [{"sourceColumnName": src_col, "targetPropertyId": str(src_prop_id)}],
        "targetKeyRefBindings": [{"sourceColumnName": tgt_col, "targetPropertyId": str(tgt_prop_id)}]
    }

# Entity definitions
hospitals_entity = entity_def(
    hosp_id, "Hospitals", "HospitalId", "HospitalName", hosp_props,
    {"HospitalId": "BigInt", "HospitalName": "String", "City": "String", "State": "String"})

departments_entity = entity_def(
    dept_id, "Departments", "DepartmentId", "DepartmentName", dept_props,
    {"DepartmentId": "BigInt", "DepartmentName": "String", "HospitalId": "BigInt", "Floor": "BigInt"})

rooms_entity = entity_def(
    room_id, "Rooms", "RoomId", "RoomNumber", room_props,
    {"RoomId": "BigInt", "RoomNumber": "String", "DepartmentId": "BigInt", "RoomType": "String"})

patients_entity = entity_def(
    pat_id, "Patients", "PatientId", "FirstName", pat_props,
    {"PatientId": "BigInt", "FirstName": "String", "LastName": "String",
     "DateOfBirth": "DateTime", "AdmissionDate": "DateTime", "CurrentRoomId": "BigInt"})

# VitalSignEquipment: static props + time-series props
equipment_entity = entity_def(
    equip_id, "VitalSignEquipment", "EquipmentId", "EquipmentId", equip_props,
    {"EquipmentId": "String", "PatientId": "BigInt", "EquipmentType": "String", "MonitoringStartDate": "DateTime"},
    ts_value_types={k: (pid, "DateTime" if k == "Timestamp" else "BigInt") for k, pid in ts_props.items()})

# Data bindings for entities
hosp_binding  = lh_binding(hosp_props,  "Hospitals")
dept_binding  = lh_binding(dept_props,  "Departments")
room_binding  = lh_binding(room_props,  "Rooms")
pat_binding   = lh_binding(pat_props,   "Patients")
equip_binding = lh_binding(equip_props, "VitalSignEquipment")  # Static binding
# Time-series binding for VitalSignEquipment from eventhouse VitalSignsReadings
equip_ts_binding = kql_ts_binding(ts_props, "VitalSignsReadings", "Timestamp", "EquipmentId", equip_props["EquipmentId"])

# Relationships
dept_hosp_rel = rel_def(rel_dept_hosp, "inHospital",        dept_id,  hosp_id)
room_dept_rel = rel_def(rel_room_dept, "inDepartment",      room_id,  dept_id)
pat_room_rel  = rel_def(rel_pat_room,  "admittedTo",        pat_id,   room_id)
equip_pat_rel = rel_def(rel_equip_pat, "assignedToPatient", equip_id, pat_id)

# Contextualizations for relationships (correct API structure)
# Format: (table, source_col, source_key_prop, target_col, target_key_prop)
dept_hosp_ctx = rel_contextualization("Departments", "DepartmentId", dept_props["DepartmentId"], "HospitalId", hosp_props["HospitalId"])
room_dept_ctx = rel_contextualization("Rooms", "RoomId", room_props["RoomId"], "DepartmentId", dept_props["DepartmentId"])
pat_room_ctx  = rel_contextualization("Patients", "PatientId", pat_props["PatientId"], "CurrentRoomId", room_props["RoomId"])
equip_pat_ctx = rel_contextualization("VitalSignEquipment", "EquipmentId", equip_props["EquipmentId"], "PatientId", pat_props["PatientId"])

print("✅ Entity and relationship definitions ready!")
print(f"   5 entities: Hospitals, Departments, Rooms, Patients, VitalSignEquipment")
print(f"   VitalSignEquipment has 2 bindings: static (lakehouse) + time-series (eventhouse)")
print(f"   4 relationships with contextualizations")


## Step 4: POST Ontology

Deletes any existing `LamnaHealthcareOntology`, then POSTs the complete definition.

In [ ]:
ontology_name = "LamnaHealthcareOntology"

token = mssparkutils.credentials.getToken("pbi")
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

# Delete existing ontology
list_resp = requests.get(
    f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items?type=Ontology",
    headers=headers
)
existing = [i for i in list_resp.json().get("value", []) if i["displayName"] == ontology_name]
for item in existing:
    del_resp = requests.delete(
        f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items/{item['id']}",
        headers=headers
    )
    print(f"🗑️  Deleted existing '{item['displayName']}' → HTTP {del_resp.status_code}")

# Assemble parts
platform_meta = {"metadata": {"type": "Ontology", "displayName": ontology_name}}

parts = [
    {"path": ".platform",      "payload": b64(platform_meta), "payloadType": "InlineBase64"},
    {"path": "definition.json","payload": b64({}),            "payloadType": "InlineBase64"},
    # Entities + bindings
    {"path": f"EntityTypes/{hosp_id}/definition.json",                          "payload": b64(hospitals_entity),  "payloadType": "InlineBase64"},
    {"path": f"EntityTypes/{hosp_id}/DataBindings/{hosp_binding['id']}.json",   "payload": b64(hosp_binding),      "payloadType": "InlineBase64"},
    {"path": f"EntityTypes/{dept_id}/definition.json",                          "payload": b64(departments_entity),"payloadType": "InlineBase64"},
    {"path": f"EntityTypes/{dept_id}/DataBindings/{dept_binding['id']}.json",   "payload": b64(dept_binding),      "payloadType": "InlineBase64"},
    {"path": f"EntityTypes/{room_id}/definition.json",                          "payload": b64(rooms_entity),      "payloadType": "InlineBase64"},
    {"path": f"EntityTypes/{room_id}/DataBindings/{room_binding['id']}.json",   "payload": b64(room_binding),      "payloadType": "InlineBase64"},
    {"path": f"EntityTypes/{pat_id}/definition.json",                           "payload": b64(patients_entity),   "payloadType": "InlineBase64"},
    {"path": f"EntityTypes/{pat_id}/DataBindings/{pat_binding['id']}.json",     "payload": b64(pat_binding),       "payloadType": "InlineBase64"},
    {"path": f"EntityTypes/{equip_id}/definition.json",                         "payload": b64(equipment_entity),  "payloadType": "InlineBase64"},
    {"path": f"EntityTypes/{equip_id}/DataBindings/{equip_binding['id']}.json", "payload": b64(equip_binding),     "payloadType": "InlineBase64"},
    {"path": f"EntityTypes/{equip_id}/DataBindings/{equip_ts_binding['id']}.json", "payload": b64(equip_ts_binding), "payloadType": "InlineBase64"},
    # Relationships + contextualizations (NOT DataBindings - this was the bug!)
    {"path": f"RelationshipTypes/{rel_dept_hosp}/definition.json", "payload": b64(dept_hosp_rel), "payloadType": "InlineBase64"},
    {"path": f"RelationshipTypes/{rel_dept_hosp}/Contextualizations/{dept_hosp_ctx['id']}.json", "payload": b64(dept_hosp_ctx), "payloadType": "InlineBase64"},
    {"path": f"RelationshipTypes/{rel_room_dept}/definition.json", "payload": b64(room_dept_rel), "payloadType": "InlineBase64"},
    {"path": f"RelationshipTypes/{rel_room_dept}/Contextualizations/{room_dept_ctx['id']}.json", "payload": b64(room_dept_ctx), "payloadType": "InlineBase64"},
    {"path": f"RelationshipTypes/{rel_pat_room}/definition.json",  "payload": b64(pat_room_rel),  "payloadType": "InlineBase64"},
    {"path": f"RelationshipTypes/{rel_pat_room}/Contextualizations/{pat_room_ctx['id']}.json",  "payload": b64(pat_room_ctx),  "payloadType": "InlineBase64"},
    {"path": f"RelationshipTypes/{rel_equip_pat}/definition.json", "payload": b64(equip_pat_rel), "payloadType": "InlineBase64"},
    {"path": f"RelationshipTypes/{rel_equip_pat}/Contextualizations/{equip_pat_ctx['id']}.json", "payload": b64(equip_pat_ctx), "payloadType": "InlineBase64"},
]

payload = {"displayName": ontology_name, "type": "Ontology", "definition": {"parts": parts}}

print(f"POSTing {len(parts)} parts for '{ontology_name}'...")
resp = requests.post(
    f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items",
    headers=headers,
    json=payload
)
print(f"Status: {resp.status_code}")

if resp.status_code == 201:
    print(f"✅ SUCCESS — ontology id: {resp.json().get('id')}")
elif resp.status_code == 202:
    location = resp.headers.get("Location")
    retry_after = int(resp.headers.get("Retry-After", 5))
    print(f"Async — polling {location}")
    for attempt in range(30):
        time.sleep(retry_after)
        poll = requests.get(location, headers=headers)
        status = poll.json().get("status")
        print(f"  attempt {attempt+1}: {status}")
        if status == "Succeeded":
            print("✅ SUCCESS")
            break
        elif status == "Failed":
            error_data = poll.json()
            print("❌ FAILED - Full response:")
            print(json.dumps(error_data, indent=2))
            break
else:
    print("❌ ERROR:", resp.text)


## Step 5: Verify

Lists the ontology in the workspace to confirm creation.

**⏳ Next steps**: After this cell completes, Fabric automatically creates the ontology and a linked Graph item that stores your entity instances. **Data ingestion time varies** — it can take anywhere from 2-20 minutes depending on capacity load, even with small datasets. To view your populated ontology with entity instances and time-series data:

1. Return to your **Fabric workspace** (in the browser)
2. Open the **LamnaHealthcareOntology** item (the Ontology item, not the Graph)
3. Select an entity type (e.g., **VitalSignEquipment** or **Patients**)
4. Click **"Entity type overview"** in the top ribbon
5. **Wait for data ingestion to complete** (check back periodically), then scroll down to see:
   - **Entity instances** (5 equipment records, patient assignments)
   - **Time-series charts** (HeartRate, OxygenSaturation, RespiratoryRate)
   - **Relationship graphs** (equipment → patient connections)

**💡 Tip**: If charts are empty after data loads, check the date filter (top-right) — change from "Last 30 days" to a custom range covering **today (April 1, 2026)** when our sample data was recorded.

In [ ]:
token = mssparkutils.credentials.getToken("pbi")
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

resp = requests.get(
    f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items?type=Ontology",
    headers=headers
)

if resp.status_code == 200:
    items = resp.json().get("value", [])
    print(f"Ontologies in workspace ({len(items)}):")
    for item in items:
        marker = " ✅" if item["displayName"] == ontology_name else ""
        print(f"  {item['displayName']}  (id: {item['id']}){marker}")
else:
    print("Could not list ontologies:", resp.status_code, resp.text)